# Predicción de temperatura a una hora

Adaptación pública del notebook operativo de temperatura: Random Forest, frecuencia de 30 minutos, ocho variables de temperatura actual/rezagada y cuatro variables cíclicas.

Se incluye un CSV **sintético** para probar el flujo. No contiene mediciones empresariales. Este notebook entrena para inferencia con el histórico disponible; no realiza una evaluación fuera de muestra ni vuelve a optimizar hiperparámetros. Las métricas laborales mencionadas en versiones previas no se publican como resultados verificados.


In [ ]:
# ============================================================
# 1. LIBRERÍAS Y CONFIGURACIÓN
# ============================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.ensemble import RandomForestRegressor

FRECUENCIA = '30min'
FECHA_INICIO_ESTABLE = pd.Timestamp('2026-02-19 13:30:00')
HORIZONTE_HORAS = 1

variables = [
    'Temp_actual',
    'Temp_30min_atras',
    'Temp_1h_atras',
    'Temp_2h_atras',
    'Temp_3h_atras',
    'Temp_6h_atras',
    'Temp_12h_atras',
    'Temp_24h_atras',
    'Hora_sin',
    'Hora_cos',
    'Dia_sin',
    'Dia_cos'
]

print("Configuración cargada correctamente")


In [ ]:
# Ruta local; en Colab puede subirse un CSV desde el panel de archivos.
from pathlib import Path
nombre_archivo = Path("datos/temperatura_sintetica.csv")
if not nombre_archivo.exists():
    try:
        from google.colab import files
        subidos = files.upload()
        nombre_archivo = Path(next(iter(subidos)))
    except ImportError as exc:
        raise FileNotFoundError("Ejecuta el notebook desde la carpeta del proyecto o ajusta nombre_archivo.") from exc
print("Archivo seleccionado:", nombre_archivo)


In [ ]:
# ============================================================
# 3. LEER Y LIMPIAR EL ARCHIVO
# ============================================================

df = pd.read_csv(
    nombre_archivo,
    sep=';',
    decimal=','
)

# Normalizamos espacios en los nombres de columnas
df.columns = [' '.join(str(col).split()) for col in df.columns]

# Buscamos automáticamente la columna de temperatura
columnas_temp = [
    col for col in df.columns
    if 'temperatura' in col.lower()
]

if len(columnas_temp) == 0:
    raise ValueError(
        "No se encontró una columna que contenga la palabra 'Temperatura'."
    )

columna_temperatura = columnas_temp[0]

df = df.rename(columns={
    columna_temperatura: 'Temperatura'
})

if 'DateTime' not in df.columns:
    raise ValueError("No se encontró la columna 'DateTime'.")

df['DateTime'] = pd.to_datetime(
    df['DateTime'],
    errors='coerce'
)

df['Temperatura'] = pd.to_numeric(
    df['Temperatura'],
    errors='coerce'
)

df = (
    df
    .dropna(subset=['DateTime', 'Temperatura'])
    .sort_values('DateTime')
    .reset_index(drop=True)
)

# Eliminamos los valores de 0 °C usados como registros inválidos
# en el histórico original
# Regla del histórico original: temperaturas <= 0 eran inválidas.
# Para otro dominio, revisar esta regla antes de usar datos reales.
df = df[df['Temperatura'] > 0].copy()
if df.empty:
    raise ValueError('El CSV no contiene mediciones válidas después de la limpieza.')

print("Registros válidos:", len(df))
print("Primera fecha:", df['DateTime'].min())
print("Última fecha:", df['DateTime'].max())
print("Última temperatura:", df.iloc[-1]['Temperatura'], "°C")


In [ ]:
# ============================================================
# 4. CONSTRUIR LA SERIE REGULAR CADA 30 MINUTOS
# ============================================================

df_modelo = df[
    df['DateTime'] >= FECHA_INICIO_ESTABLE
][['DateTime', 'Temperatura']].copy()

# Ajustamos pequeños desfases al intervalo de 30 minutos más cercano
df_modelo['DateTime'] = df_modelo['DateTime'].dt.round(FRECUENCIA)

# Conservamos el primer dato si al redondear aparece un duplicado,
# igual que durante el desarrollo original
df_modelo = (
    df_modelo
    .drop_duplicates(subset='DateTime', keep='first')
    .sort_values('DateTime')
    .set_index('DateTime')
)

rango_completo = pd.date_range(
    start=df_modelo.index.min(),
    end=df_modelo.index.max(),
    freq=FRECUENCIA
)

df_modelo = df_modelo.reindex(rango_completo)
df_modelo.index.name = 'DateTime'

print("Registros esperados en la malla de 30 min:", len(df_modelo))
print("Datos faltantes:", df_modelo['Temperatura'].isna().sum())
print("Último dato real:", df_modelo['Temperatura'].last_valid_index())


In [ ]:
# ============================================================
# 5. CREAR LAS VARIABLES DEL MODELO Y EL TARGET +1 HORA
# ============================================================

datos_ml = df_modelo.copy()

datos_ml['Temp_actual']       = datos_ml['Temperatura']
datos_ml['Temp_30min_atras']  = datos_ml['Temperatura'].shift(1)
datos_ml['Temp_1h_atras']     = datos_ml['Temperatura'].shift(2)
datos_ml['Temp_2h_atras']     = datos_ml['Temperatura'].shift(4)
datos_ml['Temp_3h_atras']     = datos_ml['Temperatura'].shift(6)
datos_ml['Temp_6h_atras']     = datos_ml['Temperatura'].shift(12)
datos_ml['Temp_12h_atras']    = datos_ml['Temperatura'].shift(24)
datos_ml['Temp_24h_atras']    = datos_ml['Temperatura'].shift(48)

hora = datos_ml.index.hour + datos_ml.index.minute / 60
dia_semana = datos_ml.index.dayofweek

datos_ml['Hora_sin'] = np.sin(2 * np.pi * hora / 24)
datos_ml['Hora_cos'] = np.cos(2 * np.pi * hora / 24)

datos_ml['Dia_sin'] = np.sin(2 * np.pi * dia_semana / 7)
datos_ml['Dia_cos'] = np.cos(2 * np.pi * dia_semana / 7)

# 1 hora = 2 pasos de 30 minutos
datos_ml['Target_1h'] = datos_ml['Temperatura'].shift(-2)

datos_entrenamiento = (
    datos_ml[variables + ['Target_1h']]
    .dropna()
    .copy()
)

X = datos_entrenamiento[variables]
y = datos_entrenamiento['Target_1h']

print("Ejemplos disponibles para entrenar:", len(X))
print("Número de variables:", X.shape[1])
print("Última fecha con target conocido:", datos_entrenamiento.index.max())


In [ ]:
# ============================================================
# 6. ENTRENAR EL MODELO DEFINITIVO
# ============================================================

# Hiperparámetros ya seleccionados durante la fase de optimización.
# No se vuelve a ejecutar RandomizedSearchCV.

modelo_final = RandomForestRegressor(
    n_estimators=500,
    max_depth=10,
    min_samples_split=10,
    min_samples_leaf=5,
    max_features=0.8,
    random_state=42,
    n_jobs=-1
)

modelo_final.fit(X, y)

print("Modelo definitivo entrenado correctamente")


In [ ]:
# ============================================================
# 7. FUNCIÓN DE PREDICCIÓN OPERATIVA +1 HORA
# ============================================================

def predecir_ultima_hora(df_historial, modelo, variables):

    serie_real = df_historial['Temperatura'].dropna()

    if serie_real.empty:
        raise ValueError("No existen mediciones reales disponibles.")

    fecha_actual = serie_real.index.max()
    temperatura_actual = float(serie_real.loc[fecha_actual])

    fechas_necesarias = {
        'Temp_actual':
            fecha_actual,

        'Temp_30min_atras':
            fecha_actual - pd.Timedelta(minutes=30),

        'Temp_1h_atras':
            fecha_actual - pd.Timedelta(hours=1),

        'Temp_2h_atras':
            fecha_actual - pd.Timedelta(hours=2),

        'Temp_3h_atras':
            fecha_actual - pd.Timedelta(hours=3),

        'Temp_6h_atras':
            fecha_actual - pd.Timedelta(hours=6),

        'Temp_12h_atras':
            fecha_actual - pd.Timedelta(hours=12),

        'Temp_24h_atras':
            fecha_actual - pd.Timedelta(hours=24)
    }

    faltan = []

    for nombre, fecha in fechas_necesarias.items():

        if fecha not in df_historial.index:
            faltan.append(fecha)

        elif pd.isna(
            df_historial.loc[fecha, 'Temperatura']
        ):
            faltan.append(fecha)

    if faltan:
        print("No se puede realizar la predicción.")
        print("Faltan estas mediciones necesarias:")

        for fecha in faltan:
            print("-", fecha)

        return None

    fila = pd.DataFrame(index=[fecha_actual])

    for nombre, fecha in fechas_necesarias.items():
        fila[nombre] = float(
            df_historial.loc[
                fecha,
                'Temperatura'
            ]
        )

    hora = (
        fecha_actual.hour
        + fecha_actual.minute / 60
    )

    dia = fecha_actual.dayofweek

    fila['Hora_sin'] = np.sin(
        2 * np.pi * hora / 24
    )

    fila['Hora_cos'] = np.cos(
        2 * np.pi * hora / 24
    )

    fila['Dia_sin'] = np.sin(
        2 * np.pi * dia / 7
    )

    fila['Dia_cos'] = np.cos(
        2 * np.pi * dia / 7
    )

    fila = fila[variables]

    temperatura_predicha = float(
        modelo.predict(fila)[0]
    )

    fecha_futura = (
        fecha_actual
        + pd.Timedelta(hours=HORIZONTE_HORAS)
    )

    diferencia = (
        temperatura_predicha
        - temperatura_actual
    )

    resultado = pd.DataFrame({
        'Concepto': [
            'Último dato disponible',
            'Temperatura actual',
            'Hora objetivo',
            'Temperatura predicha',
            'Cambio esperado'
        ],
        'Resultado': [
            fecha_actual.strftime('%d/%m/%Y %H:%M'),
            f'{temperatura_actual:.2f} °C',
            fecha_futura.strftime('%d/%m/%Y %H:%M'),
            f'{temperatura_predicha:.2f} °C',
            f'{diferencia:+.2f} °C'
        ]
    })

    print("=" * 62)
    print("PREDICCIÓN DE TEMPERATURA +1 HORA — SERIE DE EJEMPLO")
    print("=" * 62)
    print(
        f"Último dato disponible : "
        f"{fecha_actual.strftime('%d/%m/%Y %H:%M')}"
    )
    print(
        f"Temperatura actual     : "
        f"{temperatura_actual:.2f} °C"
    )
    print(
        f"Predicción para        : "
        f"{fecha_futura.strftime('%d/%m/%Y %H:%M')}"
    )
    print(
        f"Temperatura predicha   : "
        f"{temperatura_predicha:.2f} °C"
    )
    print(
        f"Cambio esperado        : "
        f"{diferencia:+.2f} °C"
    )
    print("=" * 62)

    return {
        'fecha_actual': fecha_actual,
        'fecha_futura': fecha_futura,
        'temperatura_actual': temperatura_actual,
        'temperatura_predicha': temperatura_predicha,
        'cambio_esperado': diferencia,
        'tabla': resultado
    }


In [ ]:
# ============================================================
# 8. EJECUTAR LA PREDICCIÓN
# ============================================================

resultado_prediccion = predecir_ultima_hora(
    df_modelo,
    modelo_final,
    variables
)

if resultado_prediccion is not None:
    display(resultado_prediccion['tabla'])


In [ ]:
# ============================================================
# 9. GRÁFICA RECIENTE + PREDICCIÓN
# ============================================================

if resultado_prediccion is not None:

    fecha_actual = resultado_prediccion['fecha_actual']
    fecha_futura = resultado_prediccion['fecha_futura']
    temperatura_predicha = resultado_prediccion['temperatura_predicha']

    # Últimas 48 horas
    inicio_grafica = fecha_actual - pd.Timedelta(hours=48)

    reciente = df_modelo.loc[
        inicio_grafica:fecha_actual,
        'Temperatura'
    ]

    plt.figure(figsize=(15, 5))

    plt.plot(
        reciente.index,
        reciente.values,
        label='Temperatura real',
        linewidth=2
    )

    plt.scatter(
        [fecha_futura],
        [temperatura_predicha],
        s=100,
        label='Predicción +1h',
        zorder=5
    )

    plt.plot(
        [fecha_actual, fecha_futura],
        [
            resultado_prediccion['temperatura_actual'],
            temperatura_predicha
        ],
        linestyle='--',
        linewidth=1.5
    )

    plt.title(
        'Temperatura reciente y predicción +1 hora — Serie de ejemplo'
    )

    plt.xlabel('Fecha')
    plt.ylabel('Temperatura (°C)')
    plt.legend()
    plt.grid(alpha=0.3)
    plt.show()


## Uso con otros datos

Ajusta la ruta, la fecha inicial y las reglas de limpieza al conjunto de datos. El CSV esperado usa `;` como separador, `,` como decimal y columnas `DateTime` y `Temperatura`. No se imputan huecos. La inferencia exige los rezagos necesarios; si faltan, no genera una predicción.

Antes de usar el modelo para tomar decisiones, evaluar con una separación temporal y comparar con un baseline de persistencia. La demostración sintética verifica funcionamiento, no precisión en campo.
